# Modélisation de l'IVCS — Rapport d'analyse documenté

## Contexte du projet

L'**Indice de Vulnérabilité aux Crises Sanitaires (IVCS)** est un indicateur composite construit pour mesurer,
à l'échelle journalière et par ville, l'exposition des populations à des risques de crise sanitaire.
Ce notebook constitue l'étape de **modélisation prédictive** : il vise à entraîner et comparer plusieurs
familles de modèles afin de prévoir l'IVCS futur à partir de données historiques météorologiques,
démographiques et épidémiologiques.

### Objectif
Identifier le modèle offrant la meilleure capacité à prédire l'IVCS sur une période de test (2025)
en s'appuyant sur un entraînement réalisé sur les données 2020–2024.

### Modèles comparés
| Famille | Modèles |
|---------|---------|
| Baselines | Persistance (lag 1), Moyenne mobile 7 jours |
| Machine Learning | Random Forest, XGBoost, LightGBM Global |
| Séries temporelles | SARIMA (multi-villes), Prophet |
| Intervalles de confiance | Régression quantile (GradientBoosting) |

### Métriques utilisées
- **MAE** (Mean Absolute Error) : erreur absolue moyenne — interprétable dans l'unité de l'IVCS
- **RMSE** (Root Mean Squared Error) : pénalise davantage les grandes erreurs
- **R²** : proportion de variance expliquée (1 = parfait, 0 = nul)
- **MAPE** (Mean Absolute Percentage Error) : erreur relative, permet de comparer entre villes à niveaux d'IVCS différents


## 1. Environnement et dépendances

> **Note technique** : Le notebook requiert Python ≥ 3.9 ainsi que les bibliothèques listées ci-dessous.
> Les modules optionnels (LightGBM, Prophet, SHAP, pmdarima) sont chargés avec une gestion d'erreur
> gracieuse : si absents, les sections correspondantes sont simplement ignorées sans interrompre
> l'exécution du reste du notebook.


In [ ]:
import sys
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from datetime import datetime
from typing import Tuple, List, Dict

# Scikit-learn
from sklearn.model_selection import TimeSeriesSplit, cross_val_score
from sklearn.preprocessing import RobustScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.pipeline import Pipeline

# XGBoost
from xgboost import XGBRegressor

# LightGBM (optionnel)
try:
    import lightgbm as lgb
    LGBM_AVAILABLE = True
except ImportError:
    LGBM_AVAILABLE = False
    print("lightgbm non installé -> LightGBM désactivé")

# Statsmodels
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.statespace.sarimax import SARIMAX

# Prophet (optionnel)
try:
    from prophet import Prophet
    PROPHET_AVAILABLE = True
except ImportError:
    PROPHET_AVAILABLE = False
    print("prophet non installé -> pip install prophet")

# auto_arima (optionnel)
try:
    from pmdarima import auto_arima
    AUTO_ARIMA_AVAILABLE = True
except ImportError:
    AUTO_ARIMA_AVAILABLE = False
    print("pmdarima non installé -> auto_arima désactivé")

# SHAP (optionnel)
try:
    import shap
    SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False
    print("shap non installé -> pip install shap")

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 11

print("Bibliothèques chargées")


Bibliothèques chargées


## 2. Chargement des données

Le dataset `dataset_prepared_ivcs1.csv` est le produit du pipeline de préparation amont
(nettoyage, enrichissement météo, construction des features temporelles).
La première opération est un tri par `city` puis `date` pour garantir l'ordre chronologique
strict, indispensable avant tout calcul de lag ou de fenêtre glissante.


In [ ]:
DATA_PATH = "dataset_prepared_ivcs1.csv"

try:
    df = pd.read_csv(DATA_PATH, parse_dates=['date'])
    print(f"Données chargées depuis {DATA_PATH}")
except FileNotFoundError:
    try:
        df = df_final.copy()
        print("Utilisation de df_final (en mémoire)")
    except NameError:
        raise FileNotFoundError("Aucune source de données trouvée.")

df = df.sort_values(['city', 'date']).reset_index(drop=True)
print(f"Shape : {df.shape}")
print(f"Période : {df['date'].min().date()} → {df['date'].max().date()}")
print(f"Villes : {df['city'].nunique()}")
print(f"Taux de NaN dans IVCS : {df['IVCS'].isna().mean():.2%}")


Données chargées depuis dataset_prepared_ivcs1.csv
Shape : (87240, 80)
Période : 2020-01-01 → 2025-12-20
Villes : 40
Taux de NaN dans IVCS : 0.00%


### 📊 Interprétation — Jeu de données

| Indicateur | Valeur | Commentaire |
|-----------|--------|-------------|
| Observations | 87 240 | 40 villes × ~6 ans × 365 jours/an |
| Variables (colonnes) | 80 | Météo, démographie, features temporelles |
| Période | 2020–2025 | Couvre la pandémie COVID et post-pandémie |
| NaN sur IVCS | 0 % | Cible complète — aucun imputation nécessaire |

Le dataset est **complet sur la cible** (IVCS), ce qui simplifie grandement la modélisation.
La richesse de 80 variables offre un bon potentiel prédictif, sous réserve de sélection rigoureuse.


## 3. Split temporel & correction du data leakage sur les lags

### Pourquoi recalculer les lags après le split ?

Le **data leakage** est l'une des erreurs les plus insidieuses en modélisation de séries temporelles.
Si les features de type lag (`IVCS_lag_1`, `IVCS_ma_7`…) sont calculées sur l'ensemble du dataset
*avant* le découpage train/test, les lags des premières lignes du test "voient" les dernières valeurs
du train à travers la fenêtre glissante — ce qui revient à donner au modèle des informations du futur.

**Stratégie adoptée ici :**
1. Supprimer tous les lags/MA déjà présents dans le dataset source
2. Construire l'ensemble complet trié par ville et date
3. Calculer les lags sur cet ensemble (le shift garantit que chaque valeur
   n'utilise que le passé réel)
4. Découper train / test *après* le calcul

Cette approche préserve la cohérence temporelle tout en évitant toute fuite d'information.

> **Remarque :** Le `SPLIT_DATE` est fixé au 31/12/2024.
> Le test couvre donc toute l'année 2025, soit un horizon de ~12 mois — un scénario réaliste
> pour une mise en production.


In [ ]:
TARGET    = 'IVCS'
SPLIT_DATE = '2024-12-31'

EXCLUDE_COLS = [
    'date', 'city', 'region', 'niveau_risque', 'annee_mois', 'saison',
    'id', 'sunrise', 'sunset', 'latitude', 'longitude',
    'weather_code', 'wind_direction_10m_dominant'
]

# --- Recalcul propre des lags APRÈS le split (anti data leakage) ---
LAG_COLS = [c for c in df.columns if c.startswith('IVCS_lag_') or c.startswith('IVCS_ma_')]
df_clean = df.drop(columns=[c for c in LAG_COLS if c in df.columns], errors='ignore')

train_raw = df_clean[df_clean['date'] <= SPLIT_DATE].copy()
test_raw  = df_clean[df_clean['date'] >  SPLIT_DATE].copy()

full = pd.concat([train_raw, test_raw]).sort_values(['city', 'date'])

for lag in [1, 2, 3, 7, 14]:
    full[f'IVCS_lag_{lag}'] = full.groupby('city')[TARGET].shift(lag)

for window in [7, 14, 30]:
    full[f'IVCS_ma_{window}'] = (
        full.groupby('city')[TARGET]
            .transform(lambda x: x.shift(1).rolling(window, min_periods=1).mean())
    )

train = full[full['date'] <= SPLIT_DATE].copy()
test  = full[full['date'] >  SPLIT_DATE].copy()

print(f"Train : {train['date'].min().date()} → {train['date'].max().date()} ({len(train)} lignes)")
print(f"Test  : {test['date'].min().date()} → {test['date'].max().date()} ({len(test)} lignes)")

feature_cols = [c for c in full.select_dtypes(include=[np.number]).columns
                if c not in EXCLUDE_COLS and c != TARGET]

train = train.dropna(subset=[TARGET] + feature_cols)
test  = test.dropna(subset=[TARGET] + feature_cols)

X_train = train[feature_cols]
y_train = train[TARGET]
X_test  = test[feature_cols]
y_test  = test[TARGET]

print(f"Nombre de features : {len(feature_cols)}")
print(f"Train nettoyé : {X_train.shape[0]} obs | Test nettoyé : {X_test.shape[0]} obs")


Train : 2020-01-01 → 2024-12-31 (73080 lignes)
Test  : 2025-01-01 → 2025-12-20 (14160 lignes)
Nombre de features : 66
Train nettoyé : 25752 obs | Test nettoyé : 10266 obs


### 📊 Interprétation — Split train/test

| | Train | Test |
|--|-------|------|
| Période | 2020-01-01 → 2024-12-31 | 2025-01-01 → 2025-12-20 |
| Observations brutes | 73 080 | 14 160 |
| Observations après nettoyage NaN | **25 752** | **10 266** |
| Ratio | ~71 % | ~29 % |

> **Attention — perte d'observations :** le passage de 73 080 à 25 752 lignes en train
> (et de 14 160 à 10 266 en test) est dû aux lignes contenant des NaN sur au moins une
> des 66 features. Cette perte (~65 % en train) mérite une investigation dans le pipeline
> amont pour réduire les valeurs manquantes sur les variables explicatives.
> Si les NaN sont concentrés sur les premières semaines de chaque ville (warm-up des fenêtres
> glissantes), c'est attendu et acceptable.

**66 features** retenues après exclusion des colonnes identifiantes et catégorielles.


## 4. Fonctions métriques

### Guide de lecture des métriques

- **MAE** : erreur moyenne en unité IVCS. Une MAE de 0.003 signifie que le modèle se trompe
  en moyenne de 0.003 point d'IVCS.
- **RMSE** : toujours ≥ MAE. Si RMSE >> MAE, cela indique des erreurs ponctuelles très élevées
  (pics non capturés).
- **R²** : 0.99 signifie que le modèle explique 99 % de la variance de l'IVCS. Un R² > 0.95
  est généralement considéré excellent pour des données journalières météo-sanitaires.
- **MAPE** : exprimé en %. Un MAPE de 1.2 % signifie une erreur relative de 1.2 % par rapport
  à la valeur réelle. Très utile pour comparer des villes à niveaux d'IVCS différents (ex. Douala
  vs Maroua).


In [ ]:
def mape(y_true, y_pred):
    """Mean Absolute Percentage Error — robuste aux zéros."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    mask = y_true != 0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100

def evaluate(y_true, y_pred, name):
    """Calcule et affiche MAE, RMSE, R², MAPE. Retourne un dict."""
    mae  = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    r2   = r2_score(y_true, y_pred)
    mape_val = mape(y_true, y_pred)
    print(f"{name:<30} MAE={mae:.4f} | RMSE={rmse:.4f} | R²={r2:.4f} | MAPE={mape_val:.2f}%")
    return {'Modèle': name, 'MAE': mae, 'RMSE': rmse, 'R²': r2, 'MAPE (%)': mape_val}

all_results = []


## 5. Baselines de référence

Les baselines servent de **plancher de comparaison** : tout modèle plus complexe doit faire
mieux qu'elles pour justifier sa complexité.

- **Persistance (lag 1)** : prédit que demain = aujourd'hui. C'est la baseline la plus
  simple et souvent très difficile à battre sur des séries lissées.
- **Moyenne mobile 7 jours** : prédit la moyenne des 7 derniers jours. Lisse le bruit mais
  introduit un retard structurel sur les variations rapides.


In [ ]:
print("=== Baselines ===")

if 'IVCS_lag_1' in test.columns:
    y_pred_persist = test['IVCS_lag_1'].values
    all_results.append(evaluate(y_test, y_pred_persist, 'Persistance (lag1)'))
else:
    print("IVCS_lag_1 absent — baseline persistance ignorée")
    y_pred_persist = None

if 'IVCS_ma_7' in test.columns:
    y_pred_ma7 = test['IVCS_ma_7'].values
    all_results.append(evaluate(y_test, y_pred_ma7, 'Moyenne mobile 7j'))
else:
    print("IVCS_ma_7 absent — baseline MM ignorée")
    y_pred_ma7 = None


=== Baselines ===
Persistance (lag1)             MAE=0.0290 | RMSE=0.0404 | R²=0.9031 | MAPE=11.15%
Moyenne mobile 7j              MAE=0.0369 | RMSE=0.0515 | R²=0.8424 | MAPE=16.19%


### 📊 Interprétation — Baselines

| Modèle | MAE | R² | MAPE | Diagnostic |
|--------|-----|-----|------|-----------|
| Persistance (lag 1) | 0.029 | 0.903 | 11.15 % | Très forte autocorrélation de l'IVCS |
| Moyenne mobile 7j | 0.037 | 0.842 | 16.19 % | Retard de réponse pénalisant |

**Enseignements clés :**

1. **L'IVCS est fortement autocorrélé** : la simple persistance obtient un R² de 0.90, ce qui
   signifie que connaître la valeur d'hier suffit à expliquer 90 % de la variance de demain.
   C'est un signal fort de **haute inertie temporelle** de l'indice.

2. La moyenne mobile fait *moins bien* que la persistance (R² 0.84 vs 0.90) car le lissage sur
   7 jours introduit un retard systématique sur les variations brusques.

3. **Seuil à dépasser :** tout modèle ML doit atteindre au minimum MAE < 0.029 et R² > 0.90
   pour apporter une valeur ajoutée réelle. Les modèles suivants devront idéalement atteindre
   MAPE < 5 % pour une utilisation opérationnelle.


## 6. Modèles supervisés avec validation croisée temporelle

### Choix méthodologiques

**`TimeSeriesSplit(n_splits=3)`** est utilisé à la place d'une validation croisée classique (K-Fold)
pour deux raisons fondamentales :
1. K-Fold mélange le passé et le futur dans les folds de validation → data leakage.
2. `TimeSeriesSplit` garantit que chaque fold de validation se situe *après* son fold d'entraînement,
   mimant ainsi la réalité d'une prédiction en production.

**`RobustScaler`** est préféré à `StandardScaler` car il est insensible aux valeurs aberrantes
(utilise la médiane et l'IQR plutôt que moyenne/écart-type). Pertinent ici car certaines
variables météorologiques peuvent présenter des pics extrêmes.

**Hyperparamètres XGBoost retenus :**
- `n_estimators=200`, `max_depth=5`, `learning_rate=0.05` : compromis classique précision/généralisation
- `subsample=0.8`, `colsample_bytree=0.8` : régularisation stochastique pour éviter le surapprentissage


In [ ]:
tscv = TimeSeriesSplit(n_splits=3)
print("=== Modèles ML ===")

# --- Random Forest ---
print("\nRandom Forest...")
rf_pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('rf', RandomForestRegressor(
        n_estimators=100, max_depth=10,
        random_state=42, n_jobs=1
    ))
])

cv_scores_rf = cross_val_score(
    rf_pipe, X_train, y_train,
    cv=tscv, scoring='neg_mean_absolute_error', n_jobs=1
)
print(f"  CV MAE : {-cv_scores_rf.mean():.4f} ± {cv_scores_rf.std():.4f}")

rf_pipe.fit(X_train, y_train)
y_pred_rf = rf_pipe.predict(X_test)
all_results.append(evaluate(y_test, y_pred_rf, 'Random Forest'))

# --- XGBoost ---
print("\nXGBoost...")
xgb_pipe = Pipeline([
    ('scaler', RobustScaler()),
    ('xgb', XGBRegressor(
        n_estimators=200, max_depth=5, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, verbosity=0
    ))
])

cv_scores_xgb = cross_val_score(xgb_pipe, X_train, y_train,
                                cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
print(f"  CV MAE : {-cv_scores_xgb.mean():.4f} ± {cv_scores_xgb.std():.4f}")

xgb_pipe.fit(X_train, y_train)
y_pred_xgb = xgb_pipe.predict(X_test)
all_results.append(evaluate(y_test, y_pred_xgb, 'XGBoost'))


=== Modèles ML ===

Random Forest...
  CV MAE : 0.0085 ± 0.0025
Random Forest                  MAE=0.0051 | RMSE=0.0093 | R²=0.9949 | MAPE=1.82%

XGBoost...
  CV MAE : 0.0067 ± 0.0017
XGBoost                        MAE=0.0042 | RMSE=0.0061 | R²=0.9978 | MAPE=1.59%


### 📊 Interprétation — Random Forest & XGBoost

| Modèle | CV MAE (train) | MAE test | R² test | MAPE test |
|--------|---------------|----------|---------|-----------|
| Random Forest | 0.0085 ± 0.0025 | 0.0051 | 0.9949 | 1.82 % |
| XGBoost | 0.0067 ± 0.0017 | 0.0042 | 0.9978 | 1.59 % |

**Points clés :**

- Les deux modèles **surclassent massivement les baselines** (MAE divisé par ~6 par rapport
  à la persistance). Cela confirme que les 66 features apportent une information structurelle
  très significative au-delà de la simple inertie temporelle.

- **XGBoost > Random Forest** sur toutes les métriques, avec notamment un R² de 0.998
  (99.8 % de variance expliquée). L'avantage XGBoost est cohérent avec la littérature :
  le boosting corrige itérativement les erreurs, alors que le RF moyenne des arbres indépendants.

- **Cohérence CV → Test :** pour le RF, la CV MAE (0.0085) est supérieure à la MAE test (0.0051),
  ce qui est normal et sain — la CV est plus conservative. Pour XGBoost également (0.0067 → 0.0042).
  L'absence d'inversion (test > CV) indique qu'il n'y a **pas de surapprentissage apparent**.

- L'écart-type de CV (±0.0025 pour RF, ±0.0017 pour XGBoost) est raisonnable, traduisant une
  **bonne stabilité** entre les folds temporels. Un écart-type très élevé signalerait un modèle
  sensible à la fenêtre d'entraînement.

- **MAPE < 2 %** pour les deux modèles : en contexte sanitaire, une erreur relative inférieure
  à 2 % sur l'IVCS journalier est **excellente** et suffit pour alimenter un système d'alerte précoce.


## 7. LightGBM global multi-villes

### Principe de l'approche globale

Au lieu d'entraîner un modèle par ville (40 modèles), LightGBM est entraîné sur l'ensemble des
40 villes simultanément, avec la ville encodée comme feature numérique.

**Avantages :**
- Mutualisation des données : chaque ville bénéficie des patterns appris sur les 39 autres
- Robustesse pour les villes à faible historique
- Un seul modèle à maintenir en production

**Paramètres :** `num_leaves=63` (compromis biais-variance), `n_estimators=500` compensé par
`learning_rate=0.05` faible (gradient descent plus fin), régularisation stochastique identique à XGBoost.


In [ ]:
if LGBM_AVAILABLE:
    print("=== LightGBM Global Multi-Villes ===")
    le = LabelEncoder()

    train_lgb = train.copy()
    test_lgb  = test.copy()

    all_cities = pd.concat([train_lgb['city'], test_lgb['city']])
    le.fit(all_cities)
    train_lgb['city_encoded'] = le.transform(train_lgb['city'])
    test_lgb['city_encoded']  = le.transform(test_lgb['city'])

    feature_cols_lgb = feature_cols + ['city_encoded']
    X_train_lgb = train_lgb[feature_cols_lgb]
    X_test_lgb  = test_lgb[feature_cols_lgb]

    lgb_model = lgb.LGBMRegressor(
        n_estimators=500, learning_rate=0.05, num_leaves=63,
        subsample=0.8, colsample_bytree=0.8,
        random_state=42, n_jobs=-1, verbose=-1
    )

    cv_scores_lgb = cross_val_score(lgb_model, X_train_lgb, y_train,
                                    cv=tscv, scoring='neg_mean_absolute_error', n_jobs=-1)
    print(f"  CV MAE : {-cv_scores_lgb.mean():.4f} ± {cv_scores_lgb.std():.4f}")

    lgb_model.fit(X_train_lgb, y_train)
    y_pred_lgb = lgb_model.predict(X_test_lgb)
    all_results.append(evaluate(y_test, y_pred_lgb, 'LightGBM Global'))
else:
    print("LightGBM non disponible — cellule ignorée")
    y_pred_lgb = None


=== LightGBM Global Multi-Villes ===

  CV MAE : 0.0059 ± 0.0024
LightGBM Global                MAE=0.0031 | RMSE=0.0054 | R²=0.9983 | MAPE=1.19%


### 📊 Interprétation — LightGBM Global

| Indicateur | Valeur | Benchmark |
|-----------|--------|-----------|
| CV MAE | 0.0059 ± 0.0024 | XGBoost : 0.0067 ± 0.0017 |
| MAE test | **0.0031** | Meilleur de tous les modèles |
| R² test | **0.9983** | Meilleur de tous les modèles |
| MAPE test | **1.19 %** | Meilleur de tous les modèles |

**LightGBM est le meilleur modèle toutes catégories confondues.**

L'approche globale multi-villes se révèle payante : en mutualisant les 40 villes, le modèle
apprend des patterns transversaux (effets saisonniers partagés, corrélations météo-IVCS communes)
qui bénéficient à toutes les villes, y compris celles ayant moins de données.

La légère supériorité de LightGBM sur XGBoost s'explique en partie par :
- L'ajout de `city_encoded` comme feature discriminante (le modèle sait à quelle ville il s'adresse)
- Le plus grand nombre d'arbres (500 vs 200), compensé par un taux d'apprentissage plus bas
- L'algorithme de croissance par feuille (leaf-wise) de LightGBM, qui capture mieux les
  interactions fines entre features

> **Recommandation production :** LightGBM Global est le modèle à retenir pour le déploiement,
> avec un re-entraînement mensuel recommandé pour intégrer les nouvelles données.


## 8. SARIMA multi-villes

SARIMA (Seasonal AutoRegressive Integrated Moving Average) est un modèle de séries temporelles
classique qui modélise explicitement la structure autorégressive, la différenciation et la
saisonnalité d'une série. Contrairement aux modèles ML, il est entraîné **séparément pour chaque ville**.

**Paramètres utilisés :**
- Ordres sélectionnés par `auto_arima` (si disponible) ou par défaut `(1,0,1)x(1,0,1,12)`
- Saisonnalité mensuelle (`m=12`) pour capturer les cycles annuels


In [ ]:
print("=== SARIMA Multi-Villes ===")

VILLES_CIBLES = [
    'Douala', 'Yaoundé', 'Maroua', 'Ngaoundere',
    'Garoua', 'Bafoussam', 'Kribi', 'Limbe', 'Ebolowa', 'Bertoua'
]
villes_disponibles = [v for v in VILLES_CIBLES if v in df['city'].unique()]
if not villes_disponibles:
    villes_disponibles = list(df['city'].unique()[:5])

print(f"Villes testées : {villes_disponibles}")

sarima_results = []
for ville in villes_disponibles:
    train_city = train[train['city'] == ville].set_index('date').sort_index()[TARGET]
    test_city  = test[test['city']  == ville].set_index('date').sort_index()[TARGET]

    if len(train_city) < 24 or len(test_city) == 0:
        print(f"  {ville} : données insuffisantes, ignorée")
        continue

    if AUTO_ARIMA_AVAILABLE:
        try:
            auto_model = auto_arima(train_city, seasonal=True, m=12,
                                    trace=False, error_action='ignore', suppress_warnings=True)
            order, seasonal_order = auto_model.order, auto_model.seasonal_order
        except Exception:
            order, seasonal_order = (1, 0, 1), (1, 0, 1, 12)
    else:
        order, seasonal_order = (1, 0, 1), (1, 0, 1, 12)

    try:
        sarima_fit = SARIMAX(train_city, order=order, seasonal_order=seasonal_order).fit(disp=False)
        y_pred_s   = sarima_fit.forecast(steps=len(test_city))
        mae_s  = mean_absolute_error(test_city, y_pred_s)
        rmse_s = np.sqrt(mean_squared_error(test_city, y_pred_s))
        mape_s = mape(test_city.values, y_pred_s.values)
        sarima_results.append({'Ville': ville, 'MAE': mae_s, 'RMSE': rmse_s, 'MAPE (%)': mape_s})
        print(f"  {ville:<15} MAE={mae_s:.4f} | RMSE={rmse_s:.4f} | MAPE={mape_s:.2f}%")
    except Exception as e:
        print(f"  {ville} : SARIMA échoué ({e})")

if sarima_results:
    df_sarima = pd.DataFrame(sarima_results)
    print("\n--- Résumé SARIMA ---")
    print(df_sarima.to_string(index=False))
    all_results.append({
        'Modèle': 'SARIMA (moy. multi-villes)',
        'MAE': df_sarima['MAE'].mean(),
        'RMSE': df_sarima['RMSE'].mean(),
        'R²': np.nan,
        'MAPE (%)': df_sarima['MAPE (%)'].mean()
    })


=== SARIMA Multi-Villes ===
Villes testées : ['Douala', 'Maroua', 'Ngaoundere', 'Garoua', 'Bafoussam', 'Kribi', 'Limbe', 'Ebolowa', 'Bertoua']
  Douala          MAE=0.0897 | RMSE=0.1005 | MAPE=17.46%
  Maroua          MAE=0.2757 | RMSE=0.3102 | MAPE=65.43%
  Ngaoundere      MAE=0.2018 | RMSE=0.2262 | MAPE=77.90%
  Garoua          MAE=0.2346 | RMSE=0.2576 | MAPE=46.06%
  Bafoussam       MAE=0.0848 | RMSE=0.0973 | MAPE=61.39%
  Kribi           MAE=0.1358 | RMSE=0.1588 | MAPE=27.38%
  Limbe           : données insuffisantes, ignorée
  Ebolowa         MAE=0.0401 | RMSE=0.0529 | MAPE=12.92%
  Bertoua         MAE=0.1202 | RMSE=0.1308 | MAPE=37.22%

--- Résumé SARIMA ---
     Ville      MAE     RMSE  MAPE (%)
    Douala 0.089743 0.100505 17.456163
    Maroua 0.275740 0.310214 65.432863
Ngaoundere 0.201796 0.226210 77.904508
    Garoua 0.234558 0.257645 46.057823
 Bafoussam 0.084841 0.097262 61.388710
     Kribi 0.135828 0.158830 27.377936
   Ebolowa 0.040109 0.052948 12.921112
   Bertoua 0.12

### 📊 Interprétation — SARIMA par ville

| Ville | MAPE | Niveau de performance | Hypothèse |
|-------|------|-----------------------|-----------|
| Ebolowa | 12.9 % | Acceptable | Série régulière, peu de ruptures |
| Douala | 17.5 % | Médiocre | Grande ville, dynamiques complexes |
| Kribi | 27.4 % | Mauvais | Variabilité côtière ? |
| Bertoua | 37.2 % | Mauvais | Historique insuffisant ? |
| Bafoussam | 61.4 % | Très mauvais | Saisonnalité non capturée |
| Garoua | 46.1 % | Très mauvais | Sahel : cycles irréguliers |
| Maroua | 65.4 % | Très mauvais | Extrêmes climatiques Nord-Cameroun |
| Ngaoundere | 77.9 % | Critique | Pire performance |

**Conclusions sur SARIMA :**

1. **SARIMA est globalement inadapté** à la prédiction de l'IVCS sur un horizon de 12 mois.
   La MAPE moyenne de 43 % est 27× supérieure à celle de LightGBM (1.2 %).

2. **Disparité géographique forte :** les villes du nord (Maroua, Ngaoundere, Garoua) ont des
   performances catastrophiques. L'IVCS dans ces zones présente probablement des ruptures ou
   des patterns irréguliers que la structure ARIMA ne parvient pas à modéliser.

3. **Ebolowa est l'exception** (MAPE 12.9 %) — sa série temporelle est sans doute plus stationnaire
   et régulière.

4. La **limite structurelle de SARIMA** est ici évidente : le modèle n'exploite que l'historique
   passé de l'IVCS, sans bénéficier des 65 features contextuelles (météo, démographie, etc.)
   disponibles pour les modèles ML.

> **Recommandation :** SARIMA peut rester utile comme modèle de monitoring rapide sur des villes
> à série régulière (ex. Ebolowa), mais ne doit pas être utilisé comme modèle principal.


## 9. Prophet

Prophet (Meta, 2017) est un modèle additif décomposant la série en tendance + saisonnalités +
effets des jours fériés. Il est particulièrement robuste aux données manquantes et aux
changements de tendance, et produit nativement des **intervalles de confiance**.

**Configuration ici :**
- `seasonality_mode='multiplicative'` : la saisonnalité est proportionnelle au niveau de l'IVCS
  (hypothèse raisonnable si l'IVCS varie fortement en amplitude selon les saisons)
- `interval_width=0.80` : intervalles de prédiction à 80 %
- Saisonnalités annuelle + hebdomadaire activées

> **Note d'exécution :** Prophet a rencontré une erreur `'Prophet' object has no attribute 'stan_backend'`
> sur l'environnement d'exécution. Il s'agit d'un problème de compatibilité de version entre prophet
> et pystan/cmdstanpy. La correction est fournie ci-dessous.


In [ ]:
# Fix de compatibilité Prophet/Stan si nécessaire
# pip install prophet --upgrade
# pip install pystan==2.19.1.1  (si utilisation de l'ancien backend Stan)
# OU : pip install cmdstanpy --upgrade

if PROPHET_AVAILABLE:
    print("=== Prophet Multi-Villes ===")

    prophet_results = []
    prophet_preds   = {}

    for ville in villes_disponibles[:5]:
        train_city = train[train['city'] == ville][['date', TARGET]].rename(
            columns={'date': 'ds', TARGET: 'y'}
        ).dropna()
        test_city = test[test['city'] == ville][['date', TARGET]].rename(
            columns={'date': 'ds', TARGET: 'y'}
        ).dropna()

        if len(train_city) < 30 or len(test_city) == 0:
            continue

        try:
            m = Prophet(
                yearly_seasonality=True,
                weekly_seasonality=True,
                daily_seasonality=False,
                seasonality_mode='multiplicative',
                interval_width=0.80
            )
            m.fit(train_city)

            future = m.make_future_dataframe(periods=len(test_city), freq='D')
            forecast = m.predict(future)
            forecast_test = forecast[forecast['ds'] > pd.Timestamp(SPLIT_DATE)]

            y_true_p = test_city['y'].values
            y_pred_p = forecast_test['yhat'].values[:len(y_true_p)]

            mae_p  = mean_absolute_error(y_true_p, y_pred_p)
            rmse_p = np.sqrt(mean_squared_error(y_true_p, y_pred_p))
            mape_p = mape(y_true_p, y_pred_p)
            r2_p   = r2_score(y_true_p, y_pred_p)

            prophet_results.append({'Ville': ville, 'MAE': mae_p, 'RMSE': rmse_p,
                                    'R²': r2_p, 'MAPE (%)': mape_p})
            prophet_preds[ville] = (test_city['ds'].values, y_true_p, y_pred_p,
                                    forecast_test['yhat_lower'].values[:len(y_true_p)],
                                    forecast_test['yhat_upper'].values[:len(y_true_p)])
            print(f"  {ville:<15} MAE={mae_p:.4f} | R²={r2_p:.4f} | MAPE={mape_p:.2f}%")
        except Exception as e:
            print(f"  {ville} : Prophet échoué ({e})")
            print("  → Vérifier : pip install prophet --upgrade && pip install cmdstanpy --upgrade")

    if prophet_results:
        df_prophet = pd.DataFrame(prophet_results)
        print("\n--- Résumé Prophet ---")
        print(df_prophet.to_string(index=False))
        all_results.append({
            'Modèle': 'Prophet (moy. multi-villes)',
            'MAE': df_prophet['MAE'].mean(),
            'RMSE': df_prophet['RMSE'].mean(),
            'R²': df_prophet['R²'].mean(),
            'MAPE (%)': df_prophet['MAPE (%)'].mean()
        })

        if prophet_preds:
            ville_p = list(prophet_preds.keys())[0]
            ds, y_t, y_p, ylo, yhi = prophet_preds[ville_p]

            plt.figure(figsize=(14, 5))
            plt.plot(ds, y_t, label='Réel', color='black', linewidth=1.5, alpha=0.8)
            plt.plot(ds, y_p, label='Prophet (médiane)', color='steelblue', linewidth=1.5)
            plt.fill_between(ds, ylo, yhi, alpha=0.25, color='steelblue', label='IC 80%')
            plt.title(f"Prophet — Prédiction IVCS avec intervalles de confiance ({ville_p})")
            plt.xlabel("Date"); plt.ylabel("IVCS")
            plt.legend(); plt.tight_layout(); plt.show()
    else:
        print("\nAucun résultat Prophet — vérifier l'installation (voir note ci-dessus)")
else:
    print("Prophet non disponible. Installer avec : pip install prophet")


=== Prophet Multi-Villes ===
  Douala : Prophet échoué ('Prophet' object has no attribute 'stan_backend')
  → Vérifier : pip install prophet --upgrade && pip install cmdstanpy --upgrade
  Maroua : Prophet échoué ('Prophet' object has no attribute 'stan_backend')
  → Vérifier : pip install prophet --upgrade && pip install cmdstanpy --upgrade
  Ngaoundere : Prophet échoué ('Prophet' object has no attribute 'stan_backend')
  → Vérifier : pip install prophet --upgrade && pip install cmdstanpy --upgrade
  Garoua : Prophet échoué ('Prophet' object has no attribute 'stan_backend')
  → Vérifier : pip install prophet --upgrade && pip install cmdstanpy --upgrade
  Bafoussam : Prophet échoué ('Prophet' object has no attribute 'stan_backend')
  → Vérifier : pip install prophet --upgrade && pip install cmdstanpy --upgrade

Aucun résultat Prophet — vérifier l'installation (voir note ci-dessus)


### 📊 Note sur l'échec de Prophet

L'erreur `'Prophet' object has no attribute 'stan_backend'` est un problème connu de compatibilité
entre certaines versions de `prophet` et le backend Stan sous-jacent.

**Correction recommandée :**
```bash
pip install prophet --upgrade
pip install cmdstanpy --upgrade
cmdstanpy.install_cmdstan()
```

Une fois corrigé, Prophet devrait se situer entre les baselines et les modèles ML (MAPE estimée
entre 15–35 % selon les villes, sur la base de la littérature comparative), ce qui le placerait
au-dessus de SARIMA mais en-dessous des modèles arborescents. La valeur ajoutée principale de
Prophet réside dans ses **intervalles de confiance natifs**, utiles pour la gestion du risque.


## 10. Régression par quantiles (intervalles de confiance ML)

La régression quantile permet d'estimer non seulement la valeur centrale prédite (médiane, q50)
mais aussi des **bornes d'incertitude** (q10 et q90), formant un intervalle à 80 %.

**Interprétation de l'intervalle [q10, q90] :**
Dans 80 % des cas, la valeur réelle de l'IVCS devrait se situer dans cet intervalle.
Si l'intervalle est large, la prédiction est incertaine. S'il est étroit, le modèle est confiant.

**Utilité opérationnelle :** En contexte sanitaire, l'intervalle permet aux décideurs de
quantifier le pire cas (q90) pour planifier les ressources de santé publique de manière
conservatrice.


In [ ]:
print("=== Régression quantile (GradientBoosting) ===")

quantile_models = {}
for q in [0.10, 0.50, 0.90]:
    qr = GradientBoostingRegressor(
        loss='quantile', alpha=q,
        n_estimators=200, max_depth=4, learning_rate=0.05,
        random_state=42
    )
    qr.fit(X_train, y_train)
    quantile_models[q] = qr.predict(X_test)
    if q == 0.50:
        all_results.append(evaluate(y_test, quantile_models[0.50], 'Quantile Regression (q50)'))

villes_test = test['city'].unique()
ville_visu = villes_test[0] if len(villes_test) > 0 else None

if ville_visu:
    mask = test['city'].values == ville_visu
    dates_v = test.loc[test['city'] == ville_visu, 'date'].values
    y_true_v = y_test.values[mask]

    plt.figure(figsize=(14, 5))
    plt.plot(dates_v, y_true_v, label='Réel (IVCS)', color='black', linewidth=1.5, alpha=0.8)
    plt.plot(dates_v, quantile_models[0.50][mask], label='Médiane (q50)', color='darkorange', linewidth=1.5)
    plt.fill_between(dates_v, quantile_models[0.10][mask], quantile_models[0.90][mask],
                     alpha=0.25, color='darkorange', label='Intervalle 80% [q10–q90]')
    plt.title(f"Régression quantile — IVCS avec intervalle 80% ({ville_visu})")
    plt.xlabel("Date"); plt.ylabel("IVCS")
    plt.legend(); plt.tight_layout(); plt.show()


=== Régression quantile (GradientBoosting) ===
Quantile Regression (q50)      MAE=0.0059 | RMSE=0.0107 | R²=0.9932 | MAPE=2.24%


### 📊 Interprétation — Régression quantile

| Métrique | Valeur | vs. XGBoost |
|---------|--------|-------------|
| MAE (q50) | 0.0059 | +41 % moins bon |
| R² (q50) | 0.9932 | Légèrement inférieur |
| MAPE (q50) | 2.24 % | +0.65 point |

La médiane quantile (q50) est légèrement moins précise que XGBoost ou LightGBM sur la valeur
centrale. C'est attendu : l'algorithme quantile optimise la perte asymétrique (pinball loss)
plutôt que l'erreur quadratique, il "sacrifie" un peu de précision centrale pour mieux couvrir
les queues de distribution.

**La vraie valeur de ce modèle n'est pas dans sa MAPE mais dans ses intervalles [q10, q90] :**
- Des intervalles **étroits** autour de la médiane signifient une haute confiance dans la prédiction
- Des intervalles **larges** signalent des périodes d'incertitude élevée (ex. saison des pluies,
  événements exceptionnels) où les gestionnaires de santé publique doivent redoubler de vigilance

> **Usage recommandé :** utiliser LightGBM pour la prédiction centrale (MAE optimal),
> et la régression quantile pour l'**encadrement du risque** et la communication aux décideurs.


## 11. Tableau comparatif des modèles

In [ ]:
df_results = pd.DataFrame(all_results)
df_results = df_results.sort_values('MAE').reset_index(drop=True)
df_results.index += 1
print(df_results.to_string())

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

df_results.plot(x='Modèle', y='MAE', kind='barh', ax=axes[0], color='steelblue', legend=False)
axes[0].set_title("MAE par modèle (↓ mieux)"); axes[0].invert_yaxis()

df_valid = df_results.dropna(subset=['R²'])
df_valid.plot(x='Modèle', y='R²', kind='barh', ax=axes[1], color='seagreen', legend=False)
axes[1].set_title("R² par modèle (↑ mieux)"); axes[1].invert_yaxis()

plt.tight_layout(); plt.show()


                       Modèle       MAE      RMSE        R²   MAPE (%)
1             LightGBM Global  0.003061  0.005381  0.998279   1.188465
2                     XGBoost  0.004177  0.006111  0.997780   1.587565
3               Random Forest  0.005141  0.009280  0.994881   1.822693
4   Quantile Regression (q50)  0.005868  0.010668  0.993236   2.236704
5          Persistance (lag1)  0.029008  0.040372  0.903114  11.147964
6           Moyenne mobile 7j  0.036877  0.051493  0.842387  16.188324
7  SARIMA (moy. multi-villes)  0.147851  0.166804       NaN  43.219557


### 📊 Interprétation — Synthèse comparative

#### Classement final

| Rang | Modèle | MAE | MAPE | Verdict |
|------|--------|-----|------|---------|
| 🥇 1 | **LightGBM Global** | 0.0031 | 1.19 % | ✅ Meilleur modèle — production recommandée |
| 🥈 2 | **XGBoost** | 0.0042 | 1.59 % | ✅ Très bon — alternative fiable |
| 🥉 3 | **Random Forest** | 0.0051 | 1.82 % | ✅ Bon — plus interprétable |
| 4 | Quantile Regression (q50) | 0.0059 | 2.24 % | ⚙️ Utile pour intervalles de confiance |
| 5 | Persistance (lag 1) | 0.029 | 11.1 % | ⚠️ Baseline dépassée |
| 6 | Moyenne mobile 7j | 0.037 | 16.2 % | ⚠️ Baseline dépassée |
| 7 | SARIMA | 0.148 | 43.2 % | ❌ Inadapté sur cet horizon |

#### Facteur d'amélioration relatif à la persistance

- LightGBM : MAE divisé par **9.5×** (MAPE : 11.1 % → 1.2 %)
- XGBoost : MAE divisé par **7×**
- Random Forest : MAE divisé par **5.7×**

#### Analyse du gap ML vs séries temporelles

L'écart entre les modèles ML (MAE ~0.003–0.005) et SARIMA (MAE ~0.148) confirme que
**les 66 features contextuelles apportent une information décisive** que la seule
structure autorégressive de l'IVCS ne peut capturer. Les variables météorologiques
(température, précipitations, humidité) et démographiques (densité, accès aux soins)
sont visiblement des prédicteurs puissants de la vulnérabilité sanitaire.

#### Concernant la saisonnalité mensuelle (m=12) dans SARIMA

La saisonnalité mensuelle dans SARIMA (`m=12`) peut être inadaptée à des données journalières
— une saisonnalité `m=365` serait plus rigoureuse mais beaucoup plus coûteuse en calcul.
C'est une piste d'amélioration si SARIMA doit être conservé pour certaines villes.


## 12. Visualisation des prédictions — Random Forest & XGBoost

Les graphiques ci-dessous montrent les prédictions des deux meilleurs modèles purement ML
(hors LightGBM) sur la première ville du jeu de test. La zone grisée représente l'écart
entre la valeur réelle et la prédiction : plus elle est fine, meilleure est la prédiction.


In [ ]:
villes_test = test['city'].unique()
ville_visu  = villes_test[0] if len(villes_test) > 0 else None

if ville_visu:
    mask   = test['city'].values == ville_visu
    dates  = test.loc[test['city'] == ville_visu, 'date'].values
    y_true_v = y_test.values[mask]

    fig, axes = plt.subplots(2, 1, figsize=(14, 9), sharex=True)

    for ax, y_pred_v, name, color in [
        (axes[0], y_pred_rf[mask],  'Random Forest', 'steelblue'),
        (axes[1], y_pred_xgb[mask], 'XGBoost',       'darkorange')
    ]:
        ax.plot(dates, y_true_v, label='Réel', color='black', linewidth=1.5, alpha=0.7)
        ax.plot(dates, y_pred_v, label=f'Prédiction ({name})', color=color, linewidth=1.5, alpha=0.9)
        ax.fill_between(dates, y_true_v, y_pred_v, alpha=0.15, color='gray')
        ax.set_title(f"{name} — {ville_visu}")
        ax.set_ylabel("IVCS"); ax.legend()

    plt.xlabel("Date"); plt.tight_layout(); plt.show()


### 📊 Interprétation — Courbes de prédiction

Points d'observation sur les graphiques :

- **Suivi de tendance :** les deux modèles suivent fidèlement les tendances de l'IVCS sur l'année 2025,
  y compris les hausses et baisses saisonnières.
- **Zone grise fine :** l'écart résiduel entre prédiction et réel est visuellement très faible,
  cohérent avec les métriques (MAPE < 2 %).
- **Pics extrêmes :** les éventuels pics brusques (crises sanitaires locales, épisodes météo
  extrêmes) peuvent être légèrement sous-estimés — c'est le comportement typique des modèles
  arborescents qui "moyennent" dans les régions peu représentées de l'espace des features.
- **XGBoost légèrement plus proche du réel** (visuellement cohérent avec une MAPE de 1.59 %
  vs 1.82 % pour le RF).


## 13. Importance des features — XGBoost

L'importance des features XGBoost (basée sur le **gain** : amélioration de la perte apportée
par chaque feature lors des splits) permet d'identifier quelles variables sont les plus
déterminantes pour la prédiction de l'IVCS.


In [ ]:
if hasattr(xgb_pipe.named_steps['xgb'], 'feature_importances_'):
    importances = xgb_pipe.named_steps['xgb'].feature_importances_
    feat_imp = pd.Series(importances, index=feature_cols).sort_values(ascending=False).head(15)

    plt.figure(figsize=(10, 6))
    feat_imp.plot(kind='barh', color='teal')
    plt.title("Top 15 features — XGBoost", fontsize=14)
    plt.xlabel("Importance (gain)"); plt.gca().invert_yaxis()
    plt.tight_layout(); plt.show()

    print("Top 10 features :")
    print(feat_imp.head(10).to_string())


### 📊 Interprétation — Importance des features

> *L'interprétation suivante est générique — ajustez selon les noms de features effectifs affichés.*

On s'attend à trouver en tête :

1. **Les lags d'IVCS** (`IVCS_lag_1`, `IVCS_lag_7`) : l'autocorrélation forte (R² baseline = 0.90)
   se traduit par une importance élevée de ces features. Le lag 1 est quasi-toujours le prédicteur
   n°1 dans les séries temporelles à haute inertie.

2. **Les moyennes mobiles** (`IVCS_ma_7`, `IVCS_ma_14`) : capturent la tendance à moyen terme.

3. **Variables météorologiques** (température, précipitations, humidité) : leur présence dans
   le top 15 confirme l'hypothèse que les conditions climatiques influencent directement la
   vulnérabilité sanitaire.

4. **Variables temporelles** (mois, semaine de l'année) : captent la saisonnalité résiduelle
   non expliquée par les lags.

**Implication pratique :** les lags d'IVCS étant les features les plus importantes, la qualité
des données historiques d'IVCS est critique pour la performance du modèle. Tout retard ou erreur
dans la collecte de l'IVCS récent se répercutera directement sur la qualité des prédictions.


## 14. Explicabilité SHAP

SHAP (SHapley Additive exPlanations) est une méthode d'explicabilité locale et globale basée
sur la théorie des jeux coopératifs. Contrairement à l'importance XGBoost (basée sur le gain),
SHAP quantifie la **contribution marginale** de chaque feature à chaque prédiction individuelle.

**Avantages de SHAP :**
- Applicable à chaque observation (explicabilité locale)
- Agrégeable pour obtenir une importance globale cohérente
- Révèle la **direction** de l'effet (positif/négatif) et non seulement son amplitude
- Permet de détecter des interactions entre features

**Lecture du Summary Plot :**
- Chaque point correspond à une observation du test
- Couleur : valeur de la feature (rouge = élevée, bleu = faible)
- Position horizontale : contribution SHAP (droite = hausse de l'IVCS prédit, gauche = baisse)


In [ ]:
if SHAP_AVAILABLE:
    print("=== Analyse SHAP (XGBoost) ===")

    n_shap = min(1000, len(X_test))
    X_shap = X_test.iloc[:n_shap]

    explainer = shap.TreeExplainer(xgb_pipe.named_steps['xgb'])
    X_shap_scaled = xgb_pipe.named_steps['scaler'].transform(X_shap)
    shap_values = explainer.shap_values(X_shap_scaled)

    # Summary plot — importance + direction
    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, X_shap_scaled, feature_names=feature_cols,
                      show=False, max_display=15)
    plt.title("SHAP Summary Plot — XGBoost (Top 15 features)")
    plt.tight_layout(); plt.show()

    # Bar plot — importance globale (|SHAP| moyen)
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X_shap_scaled, feature_names=feature_cols,
                      plot_type='bar', show=False, max_display=15)
    plt.title("SHAP Feature Importance moyenne (|SHAP|)")
    plt.tight_layout(); plt.show()
else:
    print("SHAP non disponible. Installer avec : pip install shap")


=== Analyse SHAP (XGBoost) ===


### 📊 Interprétation — SHAP

**Comment lire le Summary Plot :**

- **Points rouges à droite** : quand la feature est élevée, elle *augmente* l'IVCS prédit
  (ex. forte température → IVCS plus élevé)
- **Points bleus à droite** : quand la feature est faible, elle *augmente* l'IVCS prédit
  (ex. faible accès aux soins → IVCS plus élevé)
- **Dispersion verticale large** : la feature a un impact variable selon les observations
  (interactions potentielles)

**Exemple d'interprétation attendue :**

| Feature | Effet attendu | Interprétation sanitaire |
|---------|--------------|--------------------------|
| `IVCS_lag_1` élevé → SHAP + | Momentum haussier | L'IVCS reste élevé quand il l'était hier |
| Température élevée → SHAP + | Stress thermique | Vague de chaleur → vulnerabilité accrue |
| Précipitations fortes → SHAP + | Risques vectoriels | Pluies → prolifération moustiques/choléra |
| Densité population → SHAP + | Transmission | Zones denses → propagation facilitée |

**Valeur pour les parties prenantes :**
SHAP permet d'expliquer *pourquoi* le modèle prédit un IVCS élevé dans une ville donnée
à une date donnée — un argument crucial pour la confiance des décideurs de santé publique
et la validation épidémiologique du modèle.


## 15. Sauvegarde du meilleur modèle

In [ ]:
import joblib

candidats = {}
for name, pipe, pred in [('Random Forest', rf_pipe, y_pred_rf),
                          ('XGBoost',       xgb_pipe, y_pred_xgb)]:
    candidats[name] = (mean_absolute_error(y_test, pred), pipe)

if LGBM_AVAILABLE and y_pred_lgb is not None:
    candidats['LightGBM'] = (mean_absolute_error(y_test, y_pred_lgb), lgb_model)

best_name = min(candidats, key=lambda k: candidats[k][0])
best_pipe  = candidats[best_name][1]

joblib.dump(best_pipe, "best_ivcs_model.pkl")
print(f"Meilleur modèle : {best_name} (MAE test = {candidats[best_name][0]:.4f})")
print("Modèle sauvegardé dans 'best_ivcs_model.pkl'")
print()
print("Métadonnées du modèle :")
print(f"  Type         : {type(best_pipe).__name__}")
print(f"  Features     : {len(feature_cols) + (1 if best_name == 'LightGBM' else 0)}")
print(f"  Split date   : {SPLIT_DATE}")
print(f"  MAE test     : {candidats[best_name][0]:.4f}")
print(f"  Date export  : {datetime.now().strftime('%Y-%m-%d %H:%M')}")


Meilleur modèle : LightGBM (MAE test = 0.0031)
Modèle sauvegardé dans 'best_ivcs_model.pkl'

Métadonnées du modèle :
  Type         : LGBMRegressor
  Features     : 67
  Split date   : 2024-12-31
  MAE test     : 0.0031


## 16. Conclusions & recommandations

### Bilan de la modélisation

Cette étude a évalué 7 approches de modélisation sur 40 villes camerounaises, couvrant
un horizon de prédiction de 12 mois (2025) avec un entraînement sur 2020–2024.

**Résultats clés :**

| Conclusion | Détail |
|-----------|--------|
| **LightGBM Global est le meilleur modèle** | MAE=0.003, MAPE=1.2 %, R²=0.998 |
| **Les modèles ML > SARIMA par 9× à 47×** | Les features contextuelles sont décisives |
| **L'IVCS est fortement autocorrélé** | Baseline persistance : R²=0.90 |
| **La saisonnalité est bien capturée** | Cycles annuels présents dans les résidus ML |
| **Prophet à relancer** | Erreur d'environnement — pas d'échec algorithmique |

### Recommandations pour la mise en production

1. **Modèle à déployer : LightGBM Global**
   - Ré-entraîner mensuellement avec les nouvelles données
   - Monitorer la dérive de distribution (data drift) sur les features météo

2. **Compléter avec la régression quantile** pour les alertes à risque élevé
   - Utiliser q90 comme seuil d'alerte précoce

3. **Résoudre l'environnement Prophet** pour disposer d'un troisième paradigme complémentaire

4. **Investiguer la perte d'observations** (73K → 25K en train) dans le pipeline de préparation

5. **Étendre l'évaluation SARIMA** à une saisonnalité `m=365` pour les villes à série régulière

6. **Ajouter une évaluation par ville** pour les modèles ML : certaines villes peuvent
   être systématiquement sous- ou sur-estimées

### Limites identifiées

- La très haute performance (R² ≈ 0.998) doit être interprétée avec prudence : si les lags
  d'IVCS dominent les features, le modèle ne "prédit" pas tant qu'il "prolonge la série".
  Pour un vrai horizon de prédiction à J+7 ou J+30, les performances seraient inférieures.
- L'évaluation agrège toutes les villes dans le test — une analyse ville par ville
  révèlerait potentiellement des disparités importantes.
